In [4]:
%matplotlib qt

In [ ]:
# Run first in notebook:
# %matplotlib qt
#
# Install if needed:
# pip install pyserial numpy matplotlib

import serial
import serial.tools.list_ports
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import time

# -----------------------------
# Config
# -----------------------------
PORT = "/dev/cu.usbserial-110"  # change this
BAUD = 115200

PLOT_WINDOW_MS = 30_000
FUTURE_VIEW_MS = 3_000
PLOT_UPDATE_SEC = 0.05

YLIM = (-2500, 2500)  # adjust based on your filtered uV range

# -----------------------------
# Serial
# -----------------------------
ser = serial.Serial(PORT, BAUD, timeout=0.01)
time.sleep(2)
ser.reset_input_buffer()

# -----------------------------
# Buffers
# -----------------------------
data_t = deque(maxlen=12000)
data_y = deque(maxlen=12000)

trough_t = deque(maxlen=200)
trough_y = deque(maxlen=200)

pred_t = deque(maxlen=200)
pred_y = deque(maxlen=200)

stim_t = deque(maxlen=200)
stim_y = deque(maxlen=200)

latest_freq = None
latest_T = None
latest_freq_t = None

# -----------------------------
# Plot setup
# -----------------------------
plt.ion()
fig, ax = plt.subplots(figsize=(13, 6))

line_sig, = ax.plot([], [], linewidth=1.2, label="ESP32 filtered EEG")
sc_trough = ax.scatter([], [], marker="v", s=80, label="Detected trough")
sc_pred = ax.scatter([], [], marker="o", s=80, label="Predicted peak target")
sc_stim = ax.scatter([], [], marker="*", s=180, label="Actual audio stim")

ax.set_title("ESP32 closed-loop slow-wave stimulation visualization")
ax.set_xlabel("Time relative to newest sample (s)")
ax.set_ylabel("Filtered EEG (µV)")
ax.grid(True)
ax.legend(loc="upper right")
ax.set_xlim(-30, 3)
ax.set_ylim(*YLIM)

status = ax.text(
    0.01, 0.95,
    "",
    transform=ax.transAxes,
    verticalalignment="top",
    bbox=dict(boxstyle="round", alpha=0.15)
)

last_plot = time.time()

print("Reading ESP32 serial... Ctrl+C to stop.")

def update_scatter(scatter, ts, ys, current_ms):
    ts = np.array(ts)
    ys = np.array(ys)

    if len(ts) == 0:
        scatter.set_offsets(np.empty((0, 2)))
        return

    keep = (
        (ts >= current_ms - PLOT_WINDOW_MS) &
        (ts <= current_ms + FUTURE_VIEW_MS)
    )

    rel = (ts[keep] - current_ms) / 1000.0
    yy = ys[keep]

    if len(rel):
        scatter.set_offsets(np.c_[rel, yy])
    else:
        scatter.set_offsets(np.empty((0, 2)))

try:
    while True:
        line = ser.readline().decode(errors="ignore").strip()

        if line:
            parts = line.split(",")

            try:
                kind = parts[0]

                if kind == "DATA":
                    # DATA,t_ms,filtered_uV
                    t_ms = int(parts[1])
                    y = float(parts[2])
                    data_t.append(t_ms)
                    data_y.append(y)

                elif kind == "FREQ":
                    # FREQ,t_ms,freq_hz,T_ms
                    latest_freq_t = int(parts[1])
                    latest_freq = float(parts[2])
                    latest_T = float(parts[3])

                elif kind == "TROUGH":
                    # TROUGH,trough_t_ms,trough_uV,pred_peak_t_ms
                    t_ms = int(parts[1])
                    y = float(parts[2])
                    p_ms = int(parts[3])

                    trough_t.append(t_ms)
                    trough_y.append(y)

                    pred_t.append(p_ms)
                    pred_y.append(abs(y))  # mirrored peak amplitude

                elif kind == "STIM":
                    # STIM,stim_t_ms
                    t_ms = int(parts[1])
                    stim_t.append(t_ms)
                    stim_y.append(0.0)  # plot actual audio trigger on y=0 axis

            except Exception:
                pass

        now = time.time()

        if (now - last_plot) >= PLOT_UPDATE_SEC and len(data_t) > 5:
            t = np.array(data_t)
            y = np.array(data_y)

            current_ms = t[-1]

            keep = t >= current_ms - PLOT_WINDOW_MS
            rel_t = (t[keep] - current_ms) / 1000.0
            y_view = y[keep]

            line_sig.set_data(rel_t, y_view)

            update_scatter(sc_trough, trough_t, trough_y, current_ms)
            update_scatter(sc_pred, pred_t, pred_y, current_ms)
            update_scatter(sc_stim, stim_t, stim_y, current_ms)

            freq_text = "estimating..." if latest_freq is None else f"{latest_freq:.2f} Hz"
            T_text = "estimating..." if latest_T is None else f"{latest_T:.0f} ms"

            status.set_text(
                f"ESP32 30s autocorr freq: {freq_text}\n"
                f"Period T: {T_text}\n"
                f"Troughs: {len(trough_t)}\n"
                f"Predicted peaks: {len(pred_t)}\n"
                f"Audio stims: {len(stim_t)}"
            )

            fig.canvas.draw_idle()
            fig.canvas.flush_events()

            last_plot = now

except KeyboardInterrupt:
    print("Stopped.")

finally:
    ser.close()

In [ ]:
#SWITCHING TO PYQT

In [7]:
!pip install pyserial

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.9 MB/s eta 0:00:00


In [10]:
!pip install pyqtgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.8 MB/s eta 0:00:00m eta 0:00:010:00:01


In [15]:
import sys
import time
import threading
import serial
import numpy as np
from collections import deque

from PyQt5 import QtWidgets, QtCore
import pyqtgraph as pg

# -----------------------------
# Config
# -----------------------------
PORT = "/dev/cu.usbserial-0001"
BAUD = 230400

PLOT_WINDOW_MS = 30_000
FUTURE_VIEW_MS = 3_000
YLIM = (-500, 500)

MAX_POINTS = 10000
MAX_EVENTS = 300
UPDATE_MS = 25   # 40 FPS

PRINT_EVENTS = True

# -----------------------------
# Shared buffers
# -----------------------------
data_t = deque(maxlen=MAX_POINTS)
data_y = deque(maxlen=MAX_POINTS)

trough_t = deque(maxlen=MAX_EVENTS)
trough_y = deque(maxlen=MAX_EVENTS)

pred_t = deque(maxlen=MAX_EVENTS)
pred_y = deque(maxlen=MAX_EVENTS)

stim_t = deque(maxlen=MAX_EVENTS)
stim_y = deque(maxlen=MAX_EVENTS)

latest_freq = None
latest_T = None

lock = threading.Lock()
running = True

# -----------------------------
# Serial setup
# -----------------------------
ser = serial.Serial(PORT, BAUD, timeout=0.001)
time.sleep(1)
ser.reset_input_buffer()

# -----------------------------
# Serial parser thread
# -----------------------------
def process_line(line):
    global latest_freq, latest_T

    parts = line.strip().split(",")
    if len(parts) < 2:
        return

    kind = parts[0]

    try:
        with lock:
            if kind == "DATA":
                # DATA,t_ms,filtered_uV
                t_ms = int(parts[1])
                y = float(parts[2])
                data_t.append(t_ms)
                data_y.append(y)

            elif kind == "FREQ":
                # FREQ,t_ms,freq_hz,T_ms
                latest_freq = float(parts[2])
                latest_T = float(parts[3])
                if PRINT_EVENTS:
                    print(line)

            elif kind == "TROUGH":
                # TROUGH,trough_t_ms,trough_uV,pred_peak_t_ms
                t_ms = int(parts[1])
                y = float(parts[2])
                p_ms = int(parts[3])

                trough_t.append(t_ms)
                trough_y.append(y)

                pred_t.append(p_ms)
                pred_y.append(abs(y))

                if PRINT_EVENTS:
                    print(line)

            elif kind == "STIM":
                # STIM,stim_t_ms
                t_ms = int(parts[1])
                stim_t.append(t_ms)
                stim_y.append(0.0)

                if PRINT_EVENTS:
                    print(line)

            elif kind == "BOOT":
                if PRINT_EVENTS:
                    print(line)

    except Exception:
        pass

def serial_thread():
    buffer = ""

    while running:
        try:
            n = ser.in_waiting
            if n > 0:
                chunk = ser.read(n).decode(errors="ignore")
                buffer += chunk

                lines = buffer.split("\n")
                buffer = lines[-1]

                for line in lines[:-1]:
                    line = line.strip()
                    if line:
                        process_line(line)
            else:
                time.sleep(0.001)

        except Exception as e:
            print("Serial thread error:", e)
            time.sleep(0.01)

thread = threading.Thread(target=serial_thread, daemon=True)
thread.start()

# -----------------------------
# Qt / PyQtGraph setup
# -----------------------------
app = QtWidgets.QApplication(sys.argv)

win = pg.GraphicsLayoutWidget(show=True, title="ESP32 Closed-Loop Stimulation")
win.resize(1300, 650)

plot = win.addPlot(title="ESP32 real-time slow-wave stimulation visualization")
plot.setLabel("bottom", "Time relative to newest sample", units="s")
plot.setLabel("left", "Filtered EEG", units="µV")
plot.setXRange(-PLOT_WINDOW_MS / 1000, FUTURE_VIEW_MS / 1000)
plot.setYRange(*YLIM)
plot.showGrid(x=True, y=True)
plot.addLegend()

sig_curve = plot.plot([], [], pen=pg.mkPen(width=2), name="0.5–4 Hz filtered EEG")

trough_scatter = pg.ScatterPlotItem(
    size=13,
    symbol="t",
    brush=pg.mkBrush(80, 160, 255),
    pen=pg.mkPen(None),
    name="Detected trough"
)
plot.addItem(trough_scatter)

pred_scatter = pg.ScatterPlotItem(
    size=13,
    symbol="o",
    brush=pg.mkBrush(255, 170, 0),
    pen=pg.mkPen(None),
    name="Predicted peak target"
)
plot.addItem(pred_scatter)

stim_scatter = pg.ScatterPlotItem(
    size=20,
    symbol="star",
    brush=pg.mkBrush(255, 60, 60),
    pen=pg.mkPen(None),
    name="Actual audio stim"
)
plot.addItem(stim_scatter)

status = pg.TextItem("", anchor=(0, 1), color="w")
plot.addItem(status)
status.setPos(-29.5, YLIM[1] * 0.9)

# -----------------------------
# Plot update
# -----------------------------
def make_event_points(ts, ys, current_ms):
    ts = np.array(ts)
    ys = np.array(ys)

    if len(ts) == 0:
        return []

    keep = (
        (ts >= current_ms - PLOT_WINDOW_MS) &
        (ts <= current_ms + FUTURE_VIEW_MS)
    )

    rel = (ts[keep] - current_ms) / 1000.0
    yy = ys[keep]

    return [{"pos": (float(x), float(v))} for x, v in zip(rel, yy)]

def update_plot():
    with lock:
        if len(data_t) < 2:
            return

        t = np.array(data_t)
        y = np.array(data_y)

        tr_t = list(trough_t)
        tr_y = list(trough_y)

        pr_t = list(pred_t)
        pr_y = list(pred_y)

        st_t = list(stim_t)
        st_y = list(stim_y)

        freq = latest_freq
        T = latest_T

    current_ms = t[-1]

    keep = t >= current_ms - PLOT_WINDOW_MS
    rel_t = (t[keep] - current_ms) / 1000.0
    y_view = y[keep]

    sig_curve.setData(rel_t, y_view)

    trough_scatter.setData(make_event_points(tr_t, tr_y, current_ms))
    pred_scatter.setData(make_event_points(pr_t, pr_y, current_ms))
    stim_scatter.setData(make_event_points(st_t, st_y, current_ms))

    freq_text = "estimating..." if freq is None else f"{freq:.2f} Hz"
    T_text = "estimating..." if T is None else f"{T:.0f} ms"

    status.setText(
        f"Freq: {freq_text}\n"
        f"T: {T_text}\n"
        f"Troughs: {len(tr_t)}\n"
        f"Pred peaks: {len(pr_t)}\n"
        f"Audio stims: {len(st_t)}"
    )

timer = QtCore.QTimer()
timer.timeout.connect(update_plot)
timer.start(UPDATE_MS)

print("Running. Close plot window to stop.")

try:
    app.exec()
finally:
    running = False
    thread.join(timeout=1)
    ser.close()

SerialException: [Errno 16] could not open port /dev/cu.usbserial-0001: [Errno 16] Resource busy: '/dev/cu.usbserial-0001'

In [ ]:
import sys
import time
import serial
import serial.tools.list_ports
import numpy as np
from collections import deque

from PyQt5 import QtWidgets, QtCore
import pyqtgraph as pg

#THIS ONE DOES SHOW AUDIO STIM VISUAL IN LINE WITH ACTUAL SOUND

# -----------------------------
# Config
# -----------------------------
PORT = "/dev/cu.usbserial-110"   # change if needed
BAUD = 230400                    # match Serial.begin(230400)

PLOT_WINDOW_MS = 30_000
FUTURE_VIEW_MS = 3_000

YLIM = (-500, 500)

MAX_POINTS = 5000
MAX_EVENTS = 300

UPDATE_MS = 16   # ~60 FPS

# -----------------------------
# Serial
# -----------------------------
ser = serial.Serial(PORT, BAUD, timeout=0)
time.sleep(1)
ser.reset_input_buffer()

# -----------------------------
# Buffers
# -----------------------------
data_t = deque(maxlen=MAX_POINTS)
data_y = deque(maxlen=MAX_POINTS)

trough_t = deque(maxlen=MAX_EVENTS)
trough_y = deque(maxlen=MAX_EVENTS)

pred_t = deque(maxlen=MAX_EVENTS)
pred_y = deque(maxlen=MAX_EVENTS)

stim_t = deque(maxlen=MAX_EVENTS)
stim_y = deque(maxlen=MAX_EVENTS)

latest_freq = None
latest_T = None

# -----------------------------
# Qt / PyQtGraph setup
# -----------------------------
app = QtWidgets.QApplication(sys.argv)

win = pg.GraphicsLayoutWidget(show=True, title="ESP32 Closed-Loop Stimulation")
win.resize(1300, 650)

plot = win.addPlot(title="ESP32 real-time slow-wave stimulation visualization")
plot.setLabel("bottom", "Time relative to newest sample", units="s")
plot.setLabel("left", "Filtered EEG", units="µV")
plot.setXRange(-PLOT_WINDOW_MS / 1000, FUTURE_VIEW_MS / 1000)
plot.setYRange(*YLIM)
plot.showGrid(x=True, y=True)
plot.addLegend()

sig_curve = plot.plot([], [], pen=pg.mkPen(width=2), name="Filtered EEG")

trough_scatter = pg.ScatterPlotItem(
    size=12,
    symbol="t",
    brush=pg.mkBrush(80, 160, 255),
    pen=pg.mkPen(None),
    name="Detected trough"
)
plot.addItem(trough_scatter)

pred_scatter = pg.ScatterPlotItem(
    size=12,
    symbol="o",
    brush=pg.mkBrush(255, 170, 0),
    pen=pg.mkPen(None),
    name="Predicted peak target"
)
plot.addItem(pred_scatter)

stim_scatter = pg.ScatterPlotItem(
    size=18,
    symbol="star",
    brush=pg.mkBrush(255, 60, 60),
    pen=pg.mkPen(None),
    name="Actual audio stim"
)
plot.addItem(stim_scatter)

status = pg.TextItem("", anchor=(0, 1), color="w")
plot.addItem(status)
status.setPos(-29.5, YLIM[1] * 0.9)

# -----------------------------
# Serial parser
# -----------------------------
serial_buffer = ""

def process_line(line):
    global latest_freq, latest_T

    parts = line.strip().split(",")
    if len(parts) < 2:
        return

    kind = parts[0]

    try:
        if kind == "DATA":
            # DATA,t_ms,filtered_uV
            t_ms = int(parts[1])
            y = float(parts[2])
            data_t.append(t_ms)
            data_y.append(y)

        elif kind == "FREQ":
            # FREQ,t_ms,freq_hz,T_ms
            latest_freq = float(parts[2])
            latest_T = float(parts[3])

        elif kind == "TROUGH":
            # TROUGH,trough_t_ms,trough_uV,pred_peak_t_ms
            t_ms = int(parts[1])
            y = float(parts[2])
            p_ms = int(parts[3])

            trough_t.append(t_ms)
            trough_y.append(y)

            pred_t.append(p_ms)
            pred_y.append(abs(y))

        elif kind == "STIM":
            # STIM,stim_t_ms
            t_ms = int(parts[1])
            stim_t.append(t_ms)
            stim_y.append(0.0)

    except:
        pass

def read_serial_available():
    global serial_buffer

    n = ser.in_waiting
    if n <= 0:
        return

    chunk = ser.read(n).decode(errors="ignore")
    serial_buffer += chunk

    lines = serial_buffer.split("\n")
    serial_buffer = lines[-1]

    newest_data = None
    important = []

    for line in lines[:-1]:
        line = line.strip()
        if not line:
            continue

        if line.startswith("DATA"):
            newest_data = line          # keep only newest DATA
        else:
            important.append(line)     # keep all events

    if newest_data:
        process_line(newest_data)

    for line in important:
        process_line(line)
        print(line)   # terminal acts like serial monitor

# -----------------------------
# Plot update
# -----------------------------
def update_plot():
    read_serial_available()

    if len(data_t) < 2:
        return

    t = np.array(data_t)
    y = np.array(data_y)

    current_ms = t[-1]

    keep = t >= current_ms - PLOT_WINDOW_MS
    rel_t = (t[keep] - current_ms) / 1000.0
    y_view = y[keep]

    sig_curve.setData(rel_t, y_view)

    def event_points(ts, ys):
        ts = np.array(ts)
        ys = np.array(ys)

        if len(ts) == 0:
            return []

        keep = (
            (ts >= current_ms - PLOT_WINDOW_MS) &
            (ts <= current_ms + FUTURE_VIEW_MS)
        )

        rel = (ts[keep] - current_ms) / 1000.0
        yy = ys[keep]

        return [{"pos": (float(x), float(v))} for x, v in zip(rel, yy)]

    trough_scatter.setData(event_points(trough_t, trough_y))
    pred_scatter.setData(event_points(pred_t, pred_y))
    stim_scatter.setData(event_points(stim_t, stim_y))

    freq_text = "estimating..." if latest_freq is None else f"{latest_freq:.2f} Hz"
    T_text = "estimating..." if latest_T is None else f"{latest_T:.0f} ms"

    status.setText(
        f"Freq: {freq_text}\n"
        f"T: {T_text}\n"
        f"Troughs: {len(trough_t)}\n"
        f"Pred peaks: {len(pred_t)}\n"
        f"Audio stims: {len(stim_t)}"
    )

# -----------------------------
# Timer
# -----------------------------
timer = QtCore.QTimer()
timer.timeout.connect(update_plot)
timer.start(UPDATE_MS)

print("PyQtGraph ESP32 plot running. Close window to stop.")

try:
    app.exec()
finally:
    ser.close()

In [13]:
import serial
print(serial.__file__)
#print(serial.VERSION)
print(hasattr(serial, "Serial"))

/opt/anaconda3/lib/python3.12/site-packages/serial/__init__.py
True


In [14]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python
